In [12]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import copy
import collections.abc

# =====================================================================
# 0. 路径保护：自动将工作目录重置为父目录 (Res-IRF4 根目录)
# =====================================================================
if os.path.basename(os.getcwd()) == 'project':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"✅ 当前工作目录已固定为: {os.getcwd()}")
# =====================================================================

from project.utils import get_json
from project.read_input import read_inputs, parse_inputs, read_stock, read_policies
from project.building import AgentBuildings

def update_dict(d, u):
    for k, v in u.items():
        if isinstance(v, collections.abc.Mapping):
            d[k] = update_dict(d.get(k, {}), v)
        else:
            d[k] = v
    return d

# 1. 载入并正确组装配置文件
base_config_path = 'project/config/reference_hp_diff_ac_area.json'  
config_base = get_json(base_config_path)

scenario_config_path = 'project/config/config_zcl_area_roof.json'
config_scenario_file = get_json(scenario_config_path)

config = copy.deepcopy(config_base.get('base', config_base)) 
if 'base' in config_scenario_file:
    config = update_dict(config, config_scenario_file['base'])

scenario_name = 'h2a'
if scenario_name in config_scenario_file:
    config = update_dict(config, config_scenario_file[scenario_name])

def resolve_json_paths(d):
    for k, v in d.items():
        if isinstance(v, dict):
            resolve_json_paths(v)
        elif isinstance(v, str) and v.endswith('.json'):
            d[k] = get_json(v)
    return d

config = resolve_json_paths(config)

if isinstance(config.get('policies'), str) and config['policies'].endswith('.json'):
    config['policies'] = get_json(config['policies'])
if isinstance(config.get('policies'), dict) and 'policies' in config['policies']:
    config['policies'] = config['policies']['policies']

for safe_key in ['climate_zone_run', 'urban_rural', 'simple']:
    if not config.get(safe_key):  
        config[safe_key] = {}     

print(f"✅ 配置文件组装与展开完成")

# 2. 读取数据
print("\n正在读取存量和基础数据...")
stock = read_stock(config)
policies_heater, policies_insulation, taxes = read_policies(config)
inputs = read_inputs(config)
parsed_inputs = parse_inputs(inputs, taxes, config, stock)

# =====================================================================
# [关键修复] 2.5 动态加载或伪造 resources_data
# =====================================================================
try:
    # 尝试从 model 导入 (Res-IRF4 常见的全局变量存放地)
    from project.model import resources_data
except ImportError:
    try:
        # 如果都在 read_input 里，尝试从这里抓取隐藏的全局变量
        from project.read_input import resources_data
    except ImportError:
        print("⚠️ 无法自动定位 resources_data，已启用物理测试专用 Mock 数据！")
        # 捏造一个符合 __init__ 和 stock.setter 要求的字典
        resources_data = {
            'index': {
                'Energy': ['Electricity', 'Natural gas', 'Oil fuel', 'Wood fuel', 'Heating', 'Heat']
            },
            'colors': {}
        }
# =====================================================================

# 3. 实例化建筑对象
print("正在初始化建筑矩阵...")
buildings = AgentBuildings(
    stock, 
    parsed_inputs['surface'], 
    parsed_inputs['ratio_surface'], 
    parsed_inputs['efficiency'],
    parsed_inputs['income'], 
    parsed_inputs['preferences'],
    parsed_inputs['performance_insulation_renovation'],
    lifetime_heater=parsed_inputs['lifetime_heater'],
    lifetime_cooler=parsed_inputs['lifetime_cooler'],
    year=config['start'],
    climate_model=parsed_inputs['climate_model'],
    cooling_system=parsed_inputs['cooler_activation'],
    zcl_thermal_parameters=parsed_inputs['zcl_thermal_parameters'],
    cooling_price_informations=parsed_inputs['cooling_price_informations'],
    resources_data=resources_data  # <--- 安全传入
)

# =====================================================================
# 4. 🚀 核心测试：直接调用底层物理引擎！
# =====================================================================
print("\n[开始计算] 正在执行动态风速与屋顶热力学计算...")
consumption, certificate, consumption_3uses = buildings.consumption_heating(
    climate=2019,     # 强制使用 2019 年真实气象
    freq='day',       # 强制开启逐日模式
    smooth=True, 
    full_output=True
)

print("\n✅ 计算完成！")

# =====================================================================
# 5. 查看结果量级 (Ordre de grandeur)
# =====================================================================
print("\n--- 供暖能耗统计描述 (检查是否有负数或极大值) ---")
display(consumption.describe())
# 诊断 Standard Consumption (决定是否有补贴的核心)
print("\n[诊断] 计算用于经济决策的 标准能耗 (Standard Consumption) ...")
consumption_sd, _, _ = buildings.consumption_heating_store(buildings.stock.index)

print("\n--- 标准能耗统计描述 ---")
display(consumption_sd.describe())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ 当前工作目录已固定为: c:\Users\QTao\Desktop\Python codes\Res-IRF4
✅ 配置文件组装与展开完成

正在读取存量和基础数据...
file reading function is not implemented

[Info climatic data] climate=None, freq='year', smooth=False


c:\Users\QTao\Desktop\Python codes\Res-IRF4\project\read_input.py:1122: FutureWarning: DataFrame.interpolate with method=pad is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ms_heater_built = ms_heater_built.reset_index().interpolate(axis=1, method='pad').set_index(temp_idx)
c:\Users\QTao\Desktop\Python codes\Res-IRF4\project\read_input.py:1125: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inputs.update({'ms_heater_built': ms_heater_built.fillna(0)})
c:\Users\QTao\Desktop\Python codes\Res-IRF4\project\read_input.py:1161: FutureWarning: DataFrame.interpolate with method=pad is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ms_cooler_built = ms_cooler_built.reset_index().interpolate(axis

正在初始化建筑矩阵...

[Info Physics] Received roof_albedo, dynamic wind speed (v) and roof convective heat transfer (h) calculations activated!

[开始计算] 正在执行动态风速与屋顶热力学计算...

✅ 计算完成！

--- 供暖能耗统计描述 (检查是否有负数或极大值) ---


c:\Users\QTao\Desktop\Python codes\Res-IRF4\project\building.py:424: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  efficiency = to_numeric(heating_system.replace(self._efficiency))
c:\Users\QTao\Desktop\Python codes\Res-IRF4\project\thermal.py:605: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  efficiency = pd.Series(index.get_level_values('Heating system')).astype('object').replace(DHW_EFFICIENCY).set_axis(index, axis=0)
c:\Users\QTao\Desktop\Python codes\Res-IRF4\project\building.py:424: FutureWarning: Downcasting behavior in `re

count    149.000000
mean     120.079176
std      100.281206
min        5.830669
25%       38.873239
50%       83.497072
75%      177.210776
max      376.533940
dtype: float64


[诊断] 计算用于经济决策的 标准能耗 (Standard Consumption) ...

--- 标准能耗统计描述 ---


count    149.000000
mean     146.242517
std      116.142344
min       10.460584
25%       55.404151
50%      106.474765
75%      208.141119
max      443.450008
dtype: float64

In [11]:
# =====================================================================
# 终极诊断：测试“保温改造”的经济与物理计算到底崩在哪了！
# =====================================================================

print("🔍 正在强制模型执行一次 2018 年的改造计算预演...")

# 1. 强制初始化一些经济学计算必要的空结构
if not hasattr(buildings, '_renovation_store'):
    buildings._renovation_store = {}
if not hasattr(buildings, '_switch_store'):
    buildings._switch_store = {}
if not hasattr(buildings, '_adoption_store'):
    buildings._adoption_store = {}

# 2. 调用核心改造逻辑（这就是导致没人改造的罪魁祸首函数）
try:
    buildings.calculation_retrofit(step=2018)
    print("✅ 改造模块运行完毕，开始提取中间数据...")
except Exception as e:
    print(f"❌ 改造模块崩溃了: {e}")

# 3. 提取诊断数据：看看每户人家算出来的“节能量”到底是不是 NaN 或负数！
if 'consumption_saved_households' in buildings._renovation_store:
    savings = buildings._renovation_store['consumption_saved_households']
    
    print("\n--- 诊断 1：节能量 (Savings) 统计 ---")
    display(savings.describe())
    
    # 检查 NaN
    nan_count = savings.isna().sum().sum()
    print(f"⚠️ 发现 NaN (缺失值) 数量: {nan_count}")
    if nan_count > 0:
        print("🚨 破案了！Pandas 多级索引对齐失败 (嫌疑人一)，导致经济模块报错！需要修改 building.py 的重索引 (reindex) 逻辑。")
        
    # 检查 负数
    negative_count = (savings < 0).sum().sum()
    print(f"⚠️ 发现 负节能量 (保温后更费电) 数量: {negative_count}")
    if negative_count > 0:
        print("🚨 破案了！太阳能阻隔惩罚太大 (嫌疑人二)！保温反而增加了供暖需求。")
        
else:
    print("⚠️ 未能提取到改造数据，可能连第一步筛选都没过。")

# 4. 检查 Logit 算出来的总改造家庭数量
if 'renovation' in buildings._renovation_store:
    total_renovated = buildings._renovation_store['renovation'].sum().sum()
    print(f"\n📊 2018 年 Logit 模型预测的总保温改造数量: {total_renovated:,.0f} 户")

🔍 正在强制模型执行一次 2018 年的改造计算预演...
❌ 改造模块崩溃了: 'AgentBuildings' object has no attribute 'calculation_retrofit'
⚠️ 未能提取到改造数据，可能连第一步筛选都没过。
